In [ ]:
import requests
from bs4 import BeautifulSoup
import gradio as gr
import ollama
import json
import re
from datetime import datetime


C:\Users\GIGABYTE\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\GIGABYTE\AppData\Local\Temp\ipykernel_20464\2471560945.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
OLLAMA_MODEL = "qwen3:8b"

def ask_ollama(prompt: str, system: str = None) -> str:
    """Small wrapper around Ollama's chat API so every tool calls it the same way."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(model=OLLAMA_MODEL, messages=messages)
    return response["message"]["content"]

# quick check
print(ask_ollama("Reply with just the word 'ready' if you can read this."))

ready


In [ ]:
def generate_research_questions(topic):
    prompt = f"""
Generate 5 specific and useful research questions or search queries about the following topic:

{topic}

Requirements:
- Each question/query must be on a separate line.
- Do not number them.
- Do not use bullet points.
- Do not add any introduction or explanation.
- Make the questions specific enough to be useful for web research.
"""
    
    response = ask_ollama(prompt)
    
    questions = []
    for line in response.splitlines():
        line = re.sub(r"^\s*[-*•\d.)]+\s*", "", line).strip()
        if line:
            questions.append(line)
    
    return questions

In [ ]:
generate_research_questions("Solar Energy")

In [ ]:
def summarize_source(content):
    prompt = f"""
Summarize the following source in 3-5 clear sentences.

Requirements:
- Use only information explicitly stated in the source.
- Do not add outside information or make unsupported claims.
- Focus on the main ideas, important facts, and findings.
- Keep the summary concise and factual.

Source:
{content}
"""
    
    return ask_ollama(prompt).strip()

In [ ]:
test_content = """
Solar energy is a renewable source of energy that comes from sunlight.
Solar panels use photovoltaic cells to convert sunlight into electricity.
Solar power can reduce dependence on fossil fuels and lower greenhouse gas emissions.
However, solar energy production depends on sunlight and can require significant initial investment.
"""

In [ ]:
summarize_source(test_content)

In [ ]:
def compare_sources(source1, source2):
    prompt = f"""
Compare the following two sources.

Your response must contain exactly two sections:

Agreements:
- List the main points that both sources agree on.

Differences:
- List the main points where the sources disagree, differ in emphasis, or provide different information.

Do not add information that is not present in either source.

Source 1:
{source1}

Source 2:
{source2}
"""
    
    return ask_ollama(prompt).strip()

In [ ]:
source1 = """
Solar energy is a renewable source of energy.
Solar panels can reduce dependence on fossil fuels.
The main challenge is that solar power depends on sunlight.
"""

source2 = """
Solar power is a clean and renewable energy source.
Using solar panels can reduce the use of fossil fuels.
Solar energy production can be affected by weather and the availability of sunlight.
"""

In [ ]:
compare_sources(source1, source2)

In [ ]:
def generate_report(sources, topic):
    sources_text = "\n\n".join(
        f"Source {i + 1}:\n{source}" 
        for i, source in enumerate(sources)
    )

    prompt = f"""
Write a structured research report about the following topic:

{topic}

Use only the information provided in the source summaries below.
Do not invent facts or add information from outside the sources.

The report must contain exactly these sections:

1. Introduction
Briefly introduce the research topic and its context.

2. Key Findings
Present the most important findings from the sources in a clear and organized way.

3. Conclusion
Summarize the main conclusions based on the provided sources.

4. Sources
List the provided sources as Source 1, Source 2, etc.

Source summaries:
{sources_text}
"""
    
    return ask_ollama(prompt).strip()

In [ ]:
test_sources = [
    """
    Solar energy is a renewable source of energy.
    Solar panels convert sunlight into electricity.
    Solar power can reduce dependence on fossil fuels.
    """,

    """
    Solar energy is clean and renewable.
    Advances in photovoltaic technology have improved solar panel efficiency.
    Solar energy production depends on sunlight and weather conditions.
    """,

    """
    Solar power has environmental benefits because it can reduce greenhouse gas emissions.
    However, solar installations can require significant initial investment.
    """
]

In [ ]:
generate_report(test_sources, "Solar Energy")

In [2]:
llm = ChatOllama(
    model="qwen3:8b",
    temperature=0,
    # other params...
)


In [3]:
search = DuckDuckGoSearchRun()

search_tool = Tool(
name="search",
func=search.run,
description="Search the web for information",
)

In [8]:
# Test the raw DuckDuckGoSearchRun tool
result = search.run("latest news about AI agents")
print(result)

قبل يوم واحد · Comments ; AI ATTACKS! How Hackers Weaponize Artificial Intelligence. IBM Technology · 207K views ; Jensen Says “AGI Has Arrived,” OpenAI Agents Hijack a German ... قبل ٢٠ ساعة · A new OpenAI research paper shows how AI agents are transforming work, enabling longer, more complex tasks and expanding productivity across roles. ٠٤/١٢/٢٠٢٥ · Learn to build an AI news agent using Make.com and Perplexity that automatically searches the internet, filters relevant insights, and sends weekly ... قبل ٢١ ساعة · OpenAI's rogue agents used at least 10 more sites for unauthorized comms, researchers say · Traces of OpenAI agent activity found on slew of previously ... قبل يوم واحد · AI agent news for the past 7 days, updated September 9, 2026: Accenture and Google Cloud form a Gemini Enterprise business group for agentic deployments ...


Search web

In [7]:
from ddgs import DDGS

def search_web(query: str, num_results: int = 5) -> list[dict]:
    """
    Search the web for a query and return a list of results.
    Returns: [{"title": str, "url": str, "snippet": str}, ...]
    """
    results = []
    with DDGS(timeout=20) as ddgs:
        for r in ddgs.text(query, max_results=num_results, backend="duckduckgo"):
            results.append({
                "title": r.get("title", ""),
                "url": r.get("href", ""),
                "snippet": r.get("body", "")
            })
    return results

# quick test
test_results = search_web("Most wanted tech jops in 2026", num_results=3)
for r in test_results:
    print(r["title"], "-", r["url"])


DDGSException: No results found.

Page Scraper

In [ ]:
import requests
import re
from bs4 import BeautifulSoup
def scrape_page(url: str, timeout: int = 10) -> dict:
    """
    Fetch a webpage and extract clean text content.
    Returns: {"url": str, "title": str, "text": str, "success": bool}
    """
    headers = {"User-Agent": "Mozilla/5.0 (research assistant bot)"}
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
    except requests.RequestException:
        return {"url": url, "title": "", "text": "", "success": False}

    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form"]):
        tag.decompose()

    title = soup.title.string.strip() if soup.title and soup.title.string else ""

    paragraphs = soup.find_all("p")
    text = "\n".join(p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True))
    if not text:
        text = soup.get_text(separator="\n", strip=True)

    text = re.sub(r"\n{2,}", "\n", text)
    text = text[:8000]

    return {"url": url, "title": title, "text": text, "success": True}

# quick test
if test_results:
    scraped = scrape_page(test_results[0]["url"])
    print(scraped["title"])
    print(scraped["text"])



 ...


In [ ]:
research_session = {
    "topic": None,
    "questions": [],
    "sources": [],          # list of {"url", "title", "text", "summary"}
    "conversation": []       # list of {"role": "user"/"assistant", "content": str}
}


Tools Router

In [ ]:
TOOLS_DESCRIPTION = """
You have access to these tools:
1. search_web(query) - find relevant web sources on a topic
2. scrape_page(url) - extract clean text from a specific webpage
3. summarize_source(content) - summarize a piece of text
4. compare_sources(source1, source2) - compare two sources
5. generate_report(sources, topic) - create a structured report from collected sources
6. generate_research_questions(topic) - break a topic into specific search queries
7. answer_question(question) - answer a follow-up question using sources gathered so far

Given the user's message, decide which tool(s) to call and in what order.
Respond ONLY with a JSON list of steps, like:
[{"tool": "search_web", "args": {"query": "..."}}, {"tool": "scrape_page", "args": {"url": "..."}}]

If the user is just asking a question answerable from existing sources, use answer_question.
"""

def route_tool_call(user_message: str, session: dict) -> list[dict]:
    """Ask Ollama which tool(s) to call. Returns a list of {"tool": str, "args": dict} steps."""
    context = f"""Current topic: {session.get('topic')}
Number of sources collected so far: {len(session.get('sources', []))}
"""
    prompt = f"""{TOOLS_DESCRIPTION}

{context}
User message: {user_message}

Respond with ONLY the JSON list, nothing else."""

    response = ask_ollama(prompt)

    match = re.search(r"\[.*\]", response, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return []


In [ ]:
def answer_question(question: str, session: dict) -> str:
    sources_context = "\n\n".join(
        f"[{s.get('title') or s['url']}]: {s.get('summary') or s.get('text', '')[:800]}"
        for s in session.get("sources", [])
    )
    history_text = "\n".join(
        f"{turn['role'].upper()}: {turn['content']}" for turn in session.get("conversation", [])[-6:]
    )

    prompt = f"""You are a research assistant answering follow-up questions about: {session.get('topic')}

Collected research:
{sources_context[:6000]}

Recent conversation:
{history_text}

New question: {question}

Answer using the research above. If the research doesn't cover it, say so honestly."""

    answer = ask_ollama(prompt)
    session["conversation"].append({"role": "user", "content": question})
    session["conversation"].append({"role": "assistant", "content": answer})
    return answer


In [ ]:
def run_research(topic: str, num_sources: int = 3) -> dict:
    research_session["topic"] = topic
    research_session["sources"] = []
    research_session["conversation"] = []

    questions = generate_research_questions(topic)[:3]
    research_session["questions"] = questions

    seen_urls = set()
    for q in questions:
        results = search_web(q, num_results=2)
        for r in results:
            if len(research_session["sources"]) >= num_sources:
                break
            if r["url"] in seen_urls:
                continue
            seen_urls.add(r["url"])

            scraped = scrape_page(r["url"])
            if not scraped["success"] or not scraped["text"]:
                continue

            summary = summarize_source(scraped["text"])

            research_session["sources"].append({
                "url": scraped["url"],
                "title": scraped["title"] or r["title"],
                "text": scraped["text"],
                "summary": summary
            })

    return research_session

# Example run:
# session = run_research("solar energy adoption 2026", num_sources=3)
# for s in session["sources"]:
#     print(s["title"], "-", s["url"])
#     print(s["summary"])
#     print()


In [ ]:
def gradio_chat(message, history):
    global research_session

    if message.lower().startswith("research:"):
        topic = message.split(":", 1)[1].strip()
        run_research(topic, num_sources=3)
        return f"Done researching **{topic}**. Found {len(research_session['sources'])} sources. Ask a question, or type 'report'."

    if message.strip().lower() == "report":
        if not research_session["sources"]:
            return "No sources collected yet. Start with `research: <topic>` first."
        sources_text = [s.get("summary") or s.get("text", "") for s in research_session["sources"]]
        return generate_report(sources_text, research_session["topic"])

    if not research_session["sources"]:
        return "Start with `research: <your topic>` so I can gather sources first."

    return answer_question(message, research_session)


demo = gr.ChatInterface(
    fn=gradio_chat,
    title="AI Research Assistant",
    description="Type `research: <topic>` to start, then ask follow-up questions or type `report`.",
    examples=["research: solar energy adoption 2026"]
)

# demo.launch()
